In [1]:
import os
import numpy as np
import torch
import torch.nn.functional as F
import torchvision.transforms as T
import timm
from PIL import Image
%load_ext autoreload
%autoreload 2

# Make a dataloader that mixes wevrything randomly loads the images splits a train and test and computes the embeddings
from data_handler import EleHandler
from torch.utils.data import DataLoader
from torch.utils.data.sampler import RandomSampler
from pathlib import Path
from seek_code import SEEK


/data/vision/beery/scratch/antoine/micromamba/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
def closest_valid_one_hot(prob_vector):
    """
    Converts a probability tensor into the closest valid one-hot encoded representation for each attribute.
    """
    closest_one_hot = []
    index = 0

    for group in SEEK.attribute_names:
        slice_length = SEEK.lengths[group]
        prob_slice = prob_vector[:, index:index + slice_length]
        one_hot_slice = torch.zeros_like(prob_slice)
        
        max_indices = torch.argmax(prob_slice, dim=1)
        one_hot_slice[torch.arange(prob_slice.size(0)), max_indices] = 1.0
        closest_one_hot.append(one_hot_slice)
        
        index += slice_length

    return torch.cat(closest_one_hot, dim=1)

In [7]:
def calculate_attribute_accuracy(predicted, labels):
    """
    Calculates the accuracy for each attribute based on the closest valid one-hot encoded vectors
    of predicted logits and true labels, with averaging for specified pairs where needed.
    Also computes the accuracy for the entire SEEK code, counting correct only when all attributes match.
    """
    # Convert predicted logits to closest valid one-hot representation
    predicted_one_hot = closest_valid_one_hot(predicted)
    
    # Initialize accuracy tracking for all attributes, including individual and averaged
    accuracies = {name: 0 for name in SEEK.attribute_names}
    index = 0

    # Define pairs to be averaged
    average_pairs = {
        'tusk': ['right_tusk', 'left_tusk'],
        'tear_1': ['R_tear_1', 'L_tear_1'],
        'hole_1': ['R_hole_1', 'L_hole_1'],
        'tear_2': ['R_tear_2', 'L_tear_2'],
        'hole_2': ['R_hole_2', 'L_hole_2'],
        'extreme': ['right_extreme', 'left_extreme']
    }

    # Track individual accuracies for pairing
    individual_accuracies = {name: 0 for name in SEEK.attribute_names}
    batch_count = predicted.size(0)

    for name in SEEK.attribute_names:
        length = SEEK.lengths[name]
        pred_slice = predicted_one_hot[:, index:index + length]
        label_slice = labels[:, index:index + length]

        # Calculate per-sample accuracy
        pred_indices = torch.argmax(pred_slice, dim=1)
        label_indices = torch.argmax(label_slice, dim=1)
        individual_accuracies[name] = (pred_indices == label_indices).float().mean().item()

        index += length

    # Store individual accuracies for each attribute
    accuracies.update(individual_accuracies)

    # Calculate averaged accuracies for pairs and add to results
    for avg_name, pair in average_pairs.items():
        averaged_accuracy = (individual_accuracies[pair[0]] + individual_accuracies[pair[1]]) / 2
        accuracies[avg_name] = averaged_accuracy

    # Include accuracy for whole SEEK code prediction
    correct_whole_code = torch.all(predicted_one_hot == labels, dim=1).float().mean().item()
    accuracies['average_over_characters'] = sum(accuracies.values()) / len(accuracies)
    accuracies['whole_code_correct'] = correct_whole_code


    return accuracies


In [ ]:
def epoch_pass(epoch, loader, training=True):
    model.eval()  # Freeze model for inference
    layer.train() if training else layer.eval()

    total_loss = 0
    total_average_accuracy = 0
    total_whole_code_accuracy = 0
    num_batches = len(loader)
    characteristic_accuracies = {name: 0 for name in SEEK.attribute_names} if not training else None

    for batch in loader:
        images, subject_id, ele_id, identified, subject_SEEK, ele_SEEK = batch

        # Prepare labels and transfer them to GPU
        labels = torch.stack([SEEK(s).one_hot_encode() for s in subject_SEEK]).to('cuda')
        images = images.to('cuda')

        # Obtain embeddings from the frozen model
        with torch.no_grad():
            embeddings = model(images)

        # Forward pass through the trainable layer
        outputs = layer(embeddings)

        # Calculate loss
        loss = F.mse_loss(outputs, labels)
        total_loss += loss.item()

        # Calculate accuracy for average and whole-code
        predicted_SEEK = closest_valid_one_hot(outputs)
        batch_accuracies = calculate_attribute_accuracy(predicted_SEEK, labels)
        total_average_accuracy += batch_accuracies['average_over_characters']
        total_whole_code_accuracy += batch_accuracies['whole_code_correct']
        
        # Accumulate characteristic accuracies if in evaluation mode
        if not training:
            for name in SEEK.attribute_names:
                characteristic_accuracies[name] += batch_accuracies.get(name, 0)

        # Backpropagation only for the linear layer if training
        if training:
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

    # Calculate epoch-level average loss and accuracies
    average_loss = total_loss / num_batches
    average_accuracy = total_average_accuracy / num_batches
    whole_code_accuracy = total_whole_code_accuracy / num_batches
    characteristic_accuracies = {name: accuracy / num_batches for name, accuracy in characteristic_accuracies.items()}
    
    return average_loss, average_accuracy, whole_code_accuracy, characteristic_accuracies


In [ ]:

def train_concept_head(train_loader, val_loader, backbone_frozen=True, num_epochs=10):
    """
    Trains a linear layer on top of a pre-trained model to predict SEEK codes.
    """
    for epoch in range(num_epochs):
        train_loss, train_avg_acc, train_whole_code_acc, train_char_acc = epoch_pass(epoch, train_loader, training=True)
        val_loss, val_avg_acc, val_whole_code_acc, val_char_acc = epoch_pass(epoch, val_loader, training=False)
        
        print(f" Epoch {epoch + 1}/{num_epochs} - TrainLoss: {train_loss:.4f} - train_acc: {train_avg_acc * 100:.2f}% - train_acc_perfect: {train_whole_code_acc * 100:.2f}%")
        print(f"  Validation Loss: {val_loss:.4f} -  Average Concept Accuracy: {val_avg_acc * 100:.2f}% - Whole-Code Accuracy: {val_whole_code_acc * 100:.2f}%")
    
    return train_loss, train_avg_acc, train_whole_code_acc, val_loss, val_avg_acc, val_whole_code_acc

def test_concept_head(test_loader):
    """
    Tests a linear layer on top of a pre-trained model to predict SEEK codes.
    """
    test_loss, test_avg_acc, test_whole_code_acc, characteristic_accuracies = epoch_pass(-1, test_loader, training=False)
    
    print("\nTest Summary:")
    print(f"  Test Loss: {test_loss:.4f}")
    print(f"  Average Concept Accuracy: {test_avg_acc * 100:.2f}%")
    print(f"  Whole-Code Accuracy: {test_whole_code_acc * 100:.2f}%")
    
    # Detailed accuracy for each characteristic
    print("\nDetailed Characteristic Accuracies:")
    for name, accuracy in characteristic_accuracies.items():
        print(f"  {name} Accuracy: {accuracy * 100:.2f}%")
    
    return test_loss, test_avg_acc, test_whole_code_acc, characteristic_accuracies


In [34]:
# Create dataset and dataloaders
dataset = EleHandler(dataset_dir=Path('/archive/vision/beery/animal_reid/datasets/elephants_zooniverse'),
                       dictonary_path=Path('/data/vision/beery/scratch/antoine/CBM_reid/data_processing/data/out_apr2/image_dictonary.csv'),
                       transform='MegaDescriptor-elephant'
                       )


train_size = int(0.1 * len(dataset))
val_size = int(0.05 * len(dataset))
test_size = int(0.05 * len(dataset))
remaining_size = len(dataset) - train_size - val_size - test_size


train_dataset, val_dataset, test_dataset, remaining_dataset = torch.utils.data.random_split(dataset, [train_size, val_size, test_size, remaining_size])

train_loader = DataLoader(train_dataset, batch_size=512, sampler=RandomSampler(train_dataset))
val_loader = DataLoader(val_dataset, batch_size=512, sampler=RandomSampler(val_dataset))
test_loader = DataLoader(test_dataset, batch_size=512, sampler=RandomSampler(test_dataset))


model = timm.create_model("hf-hub:BVRA/MegaDescriptor-T-224", pretrained=True, num_classes=0)
state_dict = torch.load("../weights/forest_elephants-reid_weights.pt", map_location='cuda') # Using the model preatained 
model.load_state_dict(state_dict["model"]) if "optimizer" in state_dict else model.load_state_dict(state_dict)
model.to('cuda')

layer = torch.nn.Linear(768, 63).to('cuda')
optimizer = torch.optim.Adam(layer.parameters(), lr=0.001)


train_concept_head(train_loader, val_loader, backbone_frozen = True,  num_epochs=50)
test_concept_head(test_loader)

/tmp/ipykernel_184099/244930035.py:22: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  state_dict = torch.load("../weights/forest_elephants-reid_weights.pt", map_location='cud

 Epoch 1/50 - TrainLoss: 0.3812 - train_acc: 28.13% - train_acc_perfect: 0.00%
  Validation Loss: 0.2441 -  Average Concept Accuracy: 48.03% - Whole-Code Accuracy: 0.00%
 Epoch 2/50 - TrainLoss: 0.2351 - train_acc: 50.64% - train_acc_perfect: 0.00%
  Validation Loss: 0.2370 -  Average Concept Accuracy: 60.18% - Whole-Code Accuracy: 0.00%
 Epoch 3/50 - TrainLoss: 0.2308 - train_acc: 59.97% - train_acc_perfect: 0.00%
  Validation Loss: 0.2352 -  Average Concept Accuracy: 61.79% - Whole-Code Accuracy: 0.47%
 Epoch 4/50 - TrainLoss: 0.2207 - train_acc: 61.22% - train_acc_perfect: 0.00%
  Validation Loss: 0.2096 -  Average Concept Accuracy: 61.10% - Whole-Code Accuracy: 0.00%
 Epoch 5/50 - TrainLoss: 0.1936 - train_acc: 60.74% - train_acc_perfect: 0.00%
  Validation Loss: 0.1847 -  Average Concept Accuracy: 59.40% - Whole-Code Accuracy: 0.24%
 Epoch 6/50 - TrainLoss: 0.1712 - train_acc: 59.80% - train_acc_perfect: 0.00%
  Validation Loss: 0.1757 -  Average Concept Accuracy: 58.03% - Whole-C

(0.12827837467193604,
 0.6625668155876073,
 0.021176470443606377,
 {'sex': 0.9529411196708679,
  'age': 0.9505882263183594,
  'right_tusk': 0.7576470375061035,
  'left_tusk': 0.7388234734535217,
  'R_tear_1': 0.36941173672676086,
  'R_hole_1': 0.611764669418335,
  'R_tear_2': 0.5317646861076355,
  'R_hole_2': 0.6964705586433411,
  'L_tear_1': 0.34117645025253296,
  'L_hole_1': 0.6211764216423035,
  'L_tear_2': 0.5082352757453918,
  'L_hole_2': 0.677647054195404,
  'right_extreme': 0.6941176056861877,
  'left_extreme': 0.7317646741867065,
  'ear_special': 0.8658823370933533,
  'body_special': 0.8870587944984436})

In [21]:
# Create dataset and dataloaders
dataset = EleHandler(dataset_dir=Path('/archive/vision/beery/animal_reid/datasets/elephants_zooniverse'),
                       dictonary_path=Path('/data/vision/beery/scratch/antoine/CBM_reid/data_processing/data/out_apr2/image_dictonary.csv'),
                       transform='MegaDescriptor-elephant'
                       )


train_size = int(0.1 * len(dataset))
val_size = int(0.05 * len(dataset))
test_size = int(0.05 * len(dataset))
remaining_size = len(dataset) - train_size - val_size - test_size


train_dataset, val_dataset, test_dataset, remaining_dataset = torch.utils.data.random_split(dataset, [train_size, val_size, test_size, remaining_size])

train_loader = DataLoader(train_dataset, batch_size=512, sampler=RandomSampler(train_dataset))
val_loader = DataLoader(val_dataset, batch_size=512, sampler=RandomSampler(val_dataset))
test_loader = DataLoader(test_dataset, batch_size=512, sampler=RandomSampler(test_dataset))


model = timm.create_model("hf-hub:BVRA/MegaDescriptor-T-224", pretrained=True, num_classes=0)
model.load_state_dict(state_dict["model"]) if "optimizer" in state_dict else model.load_state_dict(state_dict)
model.to('cuda')

layer = torch.nn.Linear(768, 63).to('cuda')
optimizer = torch.optim.Adam(layer.parameters(), lr=0.001)


train_concept_head(train_loader, val_loader, backbone_frozen = True,  num_epochs=50)
test_concept_head(test_loader)

Epoch 0, Batch 1, Loss: 0.41620686650276184 - Average concept accuracy: 27.67% - Perfect SEEK accuracy: 0.00%
Epoch 0, Batch 2, Loss: 0.28660017251968384 - Average concept accuracy: 41.54% - Perfect SEEK accuracy: 0.00%

Epoch 0 Summary:
  Average Loss: 0.35140351951122284
  sex Epoch Accuracy: 85.89%
  age Epoch Accuracy: 38.77%
  right_tusk Epoch Accuracy: 56.06%
  left_tusk Epoch Accuracy: 54.80%
  R_tear_1 Epoch Accuracy: 23.07%
  R_hole_1 Epoch Accuracy: 32.36%
  R_tear_2 Epoch Accuracy: 22.98%
  R_hole_2 Epoch Accuracy: 27.91%
  L_tear_1 Epoch Accuracy: 23.15%
  L_hole_1 Epoch Accuracy: 25.15%
  L_tear_2 Epoch Accuracy: 12.14%
  L_hole_2 Epoch Accuracy: 35.12%
  right_extreme Epoch Accuracy: 25.56%
  left_extreme Epoch Accuracy: 29.21%
  ear_special Epoch Accuracy: 45.66%
  body_special Epoch Accuracy: 39.72%
Epoch 0, Batch 1, Loss: 0.23973096907138824 - Average concept accuracy: 50.60% - Perfect SEEK accuracy: 0.00%

Epoch 0 Summary:
  Average Loss: 0.23973096907138824
  sex Epo

(0.1325812041759491,
 {'sex': 0.9411764144897461,
  'age': 0.9505882263183594,
  'right_tusk': 0.7552940845489502,
  'left_tusk': 0.7270587682723999,
  'R_tear_1': 0.338823527097702,
  'R_hole_1': 0.5764705538749695,
  'R_tear_2': 0.5035293698310852,
  'R_hole_2': 0.6894117593765259,
  'L_tear_1': 0.3176470398902893,
  'L_hole_1': 0.5435293912887573,
  'L_tear_2': 0.4870588183403015,
  'L_hole_2': 0.6399999856948853,
  'right_extreme': 0.7082352638244629,
  'left_extreme': 0.6894117593765259,
  'ear_special': 0.8329411149024963,
  'body_special': 0.8799999356269836})

In [35]:
# Create dataset and dataloaders
dataset = EleHandler(dataset_dir=Path('/archive/vision/beery/animal_reid/datasets/elephants_zooniverse'),
                       dictonary_path=Path('/data/vision/beery/scratch/antoine/CBM_reid/data_processing/data/out_apr2/image_dictonary.csv'),
                       transform='MegaDescriptor-elephant'
                       )


train_size = int(0.1 * len(dataset))
val_size = int(0.05 * len(dataset))
test_size = int(0.05 * len(dataset))
remaining_size = len(dataset) - train_size - val_size - test_size


train_dataset, val_dataset, test_dataset, remaining_dataset = torch.utils.data.random_split(dataset, [train_size, val_size, test_size, remaining_size])

train_loader = DataLoader(train_dataset, batch_size=512, sampler=RandomSampler(train_dataset))
val_loader = DataLoader(val_dataset, batch_size=512, sampler=RandomSampler(val_dataset))
test_loader = DataLoader(test_dataset, batch_size=512, sampler=RandomSampler(test_dataset))


model = timm.create_model("hf-hub:BVRA/MegaDescriptor-T-224", pretrained=True, num_classes=0)
state_dict = torch.load("../weights/savanna_elephants_md_v2_epoch_60.pt", map_location='cuda') # Using the model preatained 
model.load_state_dict(state_dict["model"]) if "optimizer" in state_dict else model.load_state_dict(state_dict)
model.to('cuda')

layer = torch.nn.Linear(768, 63).to('cuda')
optimizer = torch.optim.Adam(layer.parameters(), lr=0.001)


train_concept_head(train_loader, val_loader, backbone_frozen = True,  num_epochs=50)
test_concept_head(test_loader)

/tmp/ipykernel_184099/2973757096.py:22: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  state_dict = torch.load("../weights/savanna_elephants_md_v2_epoch_60.pt", map_location=

 Epoch 1/50 - TrainLoss: 0.3577 - train_acc: 31.58% - train_acc_perfect: 0.00%
  Validation Loss: 0.2562 -  Average Concept Accuracy: 43.82% - Whole-Code Accuracy: 0.00%
 Epoch 2/50 - TrainLoss: 0.2403 - train_acc: 47.98% - train_acc_perfect: 0.00%
  Validation Loss: 0.2232 -  Average Concept Accuracy: 55.28% - Whole-Code Accuracy: 0.00%
 Epoch 3/50 - TrainLoss: 0.2101 - train_acc: 58.43% - train_acc_perfect: 0.00%
  Validation Loss: 0.2079 -  Average Concept Accuracy: 59.72% - Whole-Code Accuracy: 0.00%
 Epoch 4/50 - TrainLoss: 0.1962 - train_acc: 61.70% - train_acc_perfect: 0.20%
  Validation Loss: 0.2011 -  Average Concept Accuracy: 60.81% - Whole-Code Accuracy: 0.00%
 Epoch 5/50 - TrainLoss: 0.1875 - train_acc: 62.30% - train_acc_perfect: 0.39%
  Validation Loss: 0.1949 -  Average Concept Accuracy: 60.41% - Whole-Code Accuracy: 0.24%
 Epoch 6/50 - TrainLoss: 0.1767 - train_acc: 62.43% - train_acc_perfect: 0.49%
  Validation Loss: 0.1843 -  Average Concept Accuracy: 59.60% - Whole-C

(0.1435694396495819,
 0.641604252836921,
 0.021176470443606377,
 {'sex': 0.9364705681800842,
  'age': 0.9270588159561157,
  'right_tusk': 0.7905882000923157,
  'left_tusk': 0.7505881786346436,
  'R_tear_1': 0.3176470398902893,
  'R_hole_1': 0.5458823442459106,
  'R_tear_2': 0.4399999678134918,
  'R_hole_2': 0.6517646908760071,
  'L_tear_1': 0.3011764585971832,
  'L_hole_1': 0.6094117164611816,
  'L_tear_2': 0.5105882287025452,
  'L_hole_2': 0.677647054195404,
  'right_extreme': 0.7011764645576477,
  'left_extreme': 0.7152940630912781,
  'ear_special': 0.8494117259979248,
  'body_special': 0.8847058415412903})